In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Загружаем датасет
df = pd.read_csv("mushrooms.csv")

# Разделяем признаки и целевую переменную
X = df.drop("class", axis=1)
y = df["class"]

# Кодируем категориальные признаки
X = pd.get_dummies(X, dtype=int)

# Кодируем целевую переменную
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Масштабируем признаки для нейронной сети
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Выводим информацию о данных
print("Размер датасета:", df.shape)
print("Размер матрицы признаков:", X.shape)
print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)
print("Пропуски в датасете:", df.isnull().sum().sum())
print("Классы:", label_encoder.classes_)

Размер датасета: (8124, 23)
Размер матрицы признаков: (8124, 117)
Обучающая выборка: (6499, 117)
Тестовая выборка: (1625, 117)
Пропуски в датасете: 0
Классы: ['e' 'p']


In [2]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Создаём базовые модели и метамодель
stacking_model = StackingClassifier(
    estimators=[
        ("random_forest", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )),
        ("extra_trees", ExtraTreesClassifier(
            n_estimators=100,
            random_state=42
        ))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Обучаем стекинг
stacking_model.fit(X_train, y_train)

# Получаем предсказания
y_pred_stacking = stacking_model.predict(X_test)

# Оцениваем качество
stacking_accuracy = accuracy_score(y_test, y_pred_stacking)

print("Точность модели Stacking:", round(stacking_accuracy, 4))

Точность модели Stacking: 1.0


In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Создаём модель многослойного персептрона
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42,
    early_stopping=True
)

# Обучаем модель на масштабированных данных
mlp_model.fit(X_train_scaled, y_train)

# Получаем предсказания
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Оцениваем качество модели
mlp_accuracy = accuracy_score(y_test, y_pred_mlp)

print("Точность модели MLP:", round(mlp_accuracy, 4))

Точность модели MLP: 0.9988


In [4]:
import pandas as pd

# Формируем таблицу сравнения моделей
results_df = pd.DataFrame({
    "Модель": ["Stacking", "MLP"],
    "Accuracy": [stacking_accuracy, mlp_accuracy]
})

print("Сравнение моделей:")
print(results_df.round(4).to_string(index=False))

Сравнение моделей:
  Модель  Accuracy
Stacking    1.0000
     MLP    0.9988
